# Question 3: Edge-Guided U-Net Edge Generator (Step 1)

This notebook trains the 1st stage of the edge-guided completion pipeline in which we use a U-Net-style edge generator to *predict missing edge structures* from masked/corrupted edge maps. The predicted edges provide structural guidance for the image completion model in Notebook 04.

As shown in the baseline U-Net inpainting results, the model is able to fill in the missing region and recover the overall color distribution and coarse structure of the image. However, the completed region is highly blurry and lacks fine-grained visual details. In particular, the baseline U-Net struggles to reconstruct sharp boundaries, realistic textures, and detailed semantic structures because it primarily optimizes pixel-level reconstruction losses such as L1 or MSE loss.

To improve structural realism, we wanted to explore an EdgeConnect-based approach. EdgeConnect uses a two-stage framework: first predicting the missing edge structure, then using the predicted edges to guide image completion. This approach is especially useful for dog image inpainting, where missing regions may contain important semantic boundaries such as ears, legs, fur outlines, facial features, or body contours.

* Link to paper: https://arxiv.org/abs/1901.00212

Our main research question is: Does edge-guided inpainting improve structural consistency and fine-detail realism compared to the baseline U-Net model?
(instead of directly filling missing pixels, first predict the missing image structure (edges), then generate the final image conditioned on those edges)

**Notebook outline:**
* Setup and paths
* Load train / val / test split CSVs
* Create random masks
* Generate edge maps
* Build EdgeInpaintingDataset
  * **Stage 1: Edge Generator**
  * Stage 2: Completion Generator
* Patch Discriminator
* Train mini EdgeConnect GAN
* Evaluate against baseline UNet
* Visualize results
* Save model, history, metrics, and examples

https://www.kaggle.com/datasets/jessicali9530/stanford-dogs-dataset


# Environment Set Up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

## Load Data

In [ ]:
from pathlib import Path

# Main project folder
BASE_DIR = Path("/content/drive/MyDrive/Adv CV/CV Project Folder")

# Dataset folders
ANNOTATION_DIR = BASE_DIR / "Annotation"
IMAGE_DIR = BASE_DIR / "Images"

print("Annotation exists:", ANNOTATION_DIR.exists())
print("Image folder exists:", IMAGE_DIR.exists())

In [ ]:
print("Annotation contents:")
print(list(ANNOTATION_DIR.iterdir())[:10])

print("Image contents:")
print(list(IMAGE_DIR.iterdir())[:10])

In [ ]:
# define image paths
image_paths = sorted(list(IMAGE_DIR.rglob("*.jpg")))

print("Number of images:", len(image_paths))
print("Example image path:", image_paths[0])

In [ ]:
# sanity check
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(image_paths[0]).convert("RGB")

plt.imshow(img)
plt.axis("off")
plt.show()

## Model Preparation

For EdgeConnect-based GAN approach, we need the following:

* Input: masked image, masked edges, mask

* Output: completed edge map



In [ ]:
import torch
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

### Image transform

In [ ]:
# Faster training setting
# Use 128 for quick GAN/EdgeConnect-style experiments.
# You can switch back to 256 later for final-quality results if needed.
IMG_SIZE = 128

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),   # range: [0, 1]
])

### Define a random **mask** helper

We do this because for inpainting, mask convention matters where:
* 1 = visible/known area
* 0 = missing/hole area

In [ ]:
def create_random_rect_mask(image, min_size=50, max_size=120):
    _, h, w = image.shape

    mask = torch.ones((1, h, w))

    box_w = random.randint(min_size, max_size)
    box_h = random.randint(min_size, max_size)

    x1 = random.randint(0, w - box_w)
    y1 = random.randint(0, h - box_h)

    mask[:, y1:y1+box_h, x1:x1+box_w] = 0

    return mask

### Define a simple **edge** helper

This will return an edge map from the clean dog image and EdgeConnect uses edges as structural guidance.

In [ ]:
import cv2

def create_edge_map(image_tensor):
    """
    image_tensor: torch tensor, shape [3, H, W], range [0, 1]
    returns: torch tensor, shape [1, H, W], range [0, 1]
    """
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    image_np = (image_np * 255).astype(np.uint8)

    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, threshold1=100, threshold2=200)

    edges = edges.astype(np.float32) / 255.0
    edges = torch.from_numpy(edges).unsqueeze(0)

    return edges

### Prepare dataset class

In [ ]:
class DogEdgeInpaintingDataset(Dataset):
    """
    Efficient version:
    - Caches the clean image edge map after it is computed once.
    - Still creates a new random mask each time, so training keeps data augmentation.
    """
    def __init__(self, image_paths, transform=None, mask_fn=None, cache_edges=True):
        self.image_paths = image_paths
        self.transform = transform
        self.mask_fn = mask_fn
        self.cache_edges = cache_edges
        self.edge_cache = {}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        mask = self.mask_fn(image)

        # Important speedup: Canny edge detection is CPU-heavy.
        # Cache the full clean edge map once per image instead of recomputing it every epoch.
        if self.cache_edges and idx in self.edge_cache:
            edge = self.edge_cache[idx]
        else:
            edge = create_edge_map(image)
            if self.cache_edges:
                self.edge_cache[idx] = edge

        masked_image = image * mask
        masked_edge = edge * mask

        # Stage 1 edge generator input: masked image + masked edge + mask
        edge_input = torch.cat([masked_image, masked_edge, mask], dim=0)

        # Stage 2 image generator input: masked image + full edge + mask
        image_input = torch.cat([masked_image, edge, mask], dim=0)

        return {
            "image": image,
            "mask": mask,
            "masked_image": masked_image,
            "edge": edge,
            "masked_edge": masked_edge,
            "edge_input": edge_input,
            "image_input": image_input,
        }

### DataLoader sanity check

In [ ]:
# Initial sanity-check dataset using rectangle masks
dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_random_rect_mask,
    cache_edges=True
)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Faster DataLoader settings
BATCH_SIZE = 8 if torch.cuda.is_available() else 4
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

print("Batch size:", BATCH_SIZE)
print("Train batches per epoch:", len(train_loader))

Sanity check before training model:

In [ ]:
sample = dataset[0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

As shown, the mask here might be a bit too simple.

Optional: I can also increase batch size

In [ ]:
#BATCH_SIZE = 8

#train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
#val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
#test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### Improve the mask generator

To help EdgeConnect perform better, we can use irregular masks, brush strokes, random holes or larger missing regions.

In [ ]:
import cv2
import numpy as np
import random
import torch

def create_irregular_mask(image, max_strokes=4):
    _, h, w = image.shape
    mask = np.ones((h, w), np.float32)

    for _ in range(random.randint(1, max_strokes)):
        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
        x2, y2 = random.randint(0, w-1), random.randint(0, h-1)

        thickness = random.randint(6, 25)
        cv2.line(mask, (x1, y1), (x2, y2), 0, thickness)

        radius = random.randint(6, 18)
        cv2.circle(mask, (x2, y2), radius, 0, -1)

    return torch.from_numpy(mask).unsqueeze(0)

In [ ]:
# can make the mask less chaotic/random
def create_mixed_mask(image):
    if random.random() < 0.5:
        return create_random_rect_mask(image, min_size=40, max_size=100)
    else:
        return create_irregular_mask(image, max_strokes=3)

In [ ]:
# update the dataset
dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_irregular_mask
)

Sanity check again:

In [ ]:
sample = dataset[1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

I observed that some masks would fall into the background (not on the dog). Therefore I added the following setup:

In [ ]:
def mask_touches_foreground(mask, image, threshold=0.08):
    """
    Rough heuristic: checks whether the missing area overlaps with non-background-ish regions.
    Not perfect, but useful when you don't have dog segmentation masks.
    """
    missing = (mask == 0).float()
    return missing.mean().item() > threshold

### Generate masks around the image center
(Because dogs are usually near the center)

In [ ]:
def create_center_biased_irregular_mask(image, max_strokes=3):
    _, h, w = image.shape
    mask = np.ones((h, w), np.float32)

    for _ in range(random.randint(1, max_strokes)):
        cx = random.randint(w // 4, 3 * w // 4)
        cy = random.randint(h // 4, 3 * h // 4)

        x2 = min(max(cx + random.randint(-60, 60), 0), w - 1)
        y2 = min(max(cy + random.randint(-60, 60), 0), h - 1)

        thickness = random.randint(6, 22)
        cv2.line(mask, (cx, cy), (x2, y2), 0, thickness)

        radius = random.randint(6, 18)
        cv2.circle(mask, (x2, y2), radius, 0, -1)

    return torch.from_numpy(mask).unsqueeze(0)

In [ ]:
def create_mixed_mask(image):
    r = random.random()

    if r < 0.35:
        return create_random_rect_mask(image, min_size=40, max_size=100)
    elif r < 0.80:
        return create_center_biased_irregular_mask(image, max_strokes=3)
    else:
        return create_irregular_mask(image, max_strokes=3)

Sanity check:

In [ ]:
# Final efficient dataset + dataloaders using the mixed/center-biased mask generator

dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_mixed_mask,
    cache_edges=True
)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 8 if torch.cuda.is_available() else 4
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print("Dataset size:", len(dataset))
print("Train / Val / Test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("Train batches per epoch:", len(train_loader))

In [ ]:
sample = dataset[0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

This is slightly better, so we can proceed with this mask generator.

## Train a simple U-Net Style Edge generator

Current setup:
* edge_input  # [masked image, masked edge, mask], 5 channels
* edge        # target full edge map, 1 channel

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleEdgeUNet(nn.Module):
    """
    Lighter U-Net for faster training.
    base_channels=32 is much faster than the original 64-channel version.
    If you want better final quality later, try base_channels=48 or 64.
    """
    def __init__(self, in_channels=5, out_channels=1, base_channels=32):
        super().__init__()

        c = base_channels
        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(c * 4, c * 8)

        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.final = nn.Conv2d(c, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Keep sigmoid here because the notebook uses BCELoss below.
        return torch.sigmoid(self.final(d1))

### Define model & Test model shape

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Lighter model = faster epochs
edge_generator = SimpleEdgeUNet(in_channels=5, out_channels=1, base_channels=32).to(device)

batch = next(iter(train_loader))
edge_input = batch["edge_input"].to(device, non_blocking=True)

with torch.no_grad():
    pred_edge = edge_generator(edge_input)

print("Input shape:", edge_input.shape)
print("Predicted edge shape:", pred_edge.shape)

### Define loss & optimizer

In [ ]:
import torch.nn as nn
import torch.optim as optim

bce_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()

optimizer = optim.Adam(edge_generator.parameters(), lr=1e-4)

# Mixed precision speeds up CUDA training and saves memory.
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

### Train edge generator

In [ ]:
def train_edge_generator(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=5,
    max_train_batches=None,
    max_val_batches=50,
    val_interval=1,
):
    """
    Faster training loop:
    - mixed precision on GPU
    - non_blocking GPU transfer
    - optional max_train_batches for quick experiments
    - optional max_val_batches so validation does not take forever
    """
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        train_steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - train")
        for step, batch in enumerate(pbar):
            if max_train_batches is not None and step >= max_train_batches:
                break

            edge_input = batch["edge_input"].to(device, non_blocking=True)
            target_edge = batch["edge"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred_edge = model(edge_input)

                # focus more on missing region
                hole = 1 - mask

                loss_bce = bce_loss(pred_edge, target_edge)
                loss_hole = l1_loss(pred_edge * hole, target_edge * hole)

                loss = loss_bce + 5 * loss_hole

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()
            train_steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / max(train_steps, 1)
        train_losses.append(avg_train_loss)

        # Validate less expensively
        if (epoch + 1) % val_interval == 0:
            model.eval()
            total_val_loss = 0
            val_steps = 0

            with torch.no_grad():
                for step, batch in enumerate(tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - val")):
                    if max_val_batches is not None and step >= max_val_batches:
                        break

                    edge_input = batch["edge_input"].to(device, non_blocking=True)
                    target_edge = batch["edge"].to(device, non_blocking=True)
                    mask = batch["mask"].to(device, non_blocking=True)

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        pred_edge = model(edge_input)
                        hole = 1 - mask

                        loss_bce = bce_loss(pred_edge, target_edge)
                        loss_hole = l1_loss(pred_edge * hole, target_edge * hole)

                        loss = loss_bce + 5 * loss_hole

                    total_val_loss += loss.item()
                    val_steps += 1

            avg_val_loss = total_val_loss / max(val_steps, 1)
        else:
            avg_val_loss = None

        val_losses.append(avg_val_loss)

        if avg_val_loss is None:
            print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}")
        else:
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    return train_losses, val_losses

### Run training

In [ ]:
# Efficient run-training setting
# Start with fewer epochs and a capped number of batches so you can verify the model quickly.
# After it works, remove max_train_batches or increase epochs.

edge_train_losses, edge_val_losses = train_edge_generator(
    edge_generator,
    train_loader,
    val_loader,
    optimizer,
    epochs=5,
    max_train_batches=200,   # set to None for full training
    max_val_batches=50,
    val_interval=1
)

### Visualize prediction
* Edge completion results for masked dog images.

In [ ]:
def show_edge_prediction(model, dataset, idx=0):
    model.eval()
    sample = dataset[idx]

    edge_input = sample["edge_input"].unsqueeze(0).to(device)

    with torch.no_grad():
        pred_edge = model(edge_input).cpu().squeeze()

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))

    axes[0].imshow(sample["masked_image"].permute(1, 2, 0))
    axes[0].set_title("Masked Image")

    axes[1].imshow(sample["masked_edge"].squeeze(), cmap="gray")
    axes[1].set_title("Masked Edge")

    axes[2].imshow(pred_edge, cmap="gray")
    axes[2].set_title("Predicted Edge")

    axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
    axes[3].set_title("Target Edge")

    for ax in axes:
        ax.axis("off")

    plt.show()

In [ ]:
show_edge_prediction(edge_generator, val_dataset, idx=0)

From this we can see that the edge completion network successfully reconstructs structural contours within occluded regions and provides meaningful guidance for downstream image inpainting.

The model reconstructs missing structural contours and facial boundaries from partially observed edge maps, supporting downstream image inpainting

Save the model & work

In [ ]:
torch.save(edge_generator.state_dict(), BASE_DIR / "edge_generator_unet.pt")

Edge Generator Evaluation Conclusion: For this stage, visual quality is especially important. A good edge generator should recover major dog contours, facial boundaries, and body structure in the masked region.